In [ ]:
# -*- coding: utf-8 -*-

TubesDasiltdat.ipynb

Automatically generated by Colab.

Original file is located at
https://colab.research.google.com/drive/1mupgSBSCp5QsEev0WUTEHtw_6Ydl5UDR

# 🌿 PlantSense AI - Alur Pelatihan Model (Training Pipeline)
### Sistem Klasifikasi Tingkat Stres Tanaman Berbasis Web Menggunakan Machine Learning

Notebook ini berisi alur kerja (pipeline) lengkap langkah demi langkah untuk sistem **PlantSense AI**, yang mengklasifikasikan status kesehatan tanaman (**Healthy**, **Moderate Stress**, dan **High Stress**) berdasarkan data biosensor dan kandungan klorofil.

### 🧑‍🤝‍🧑 Anggota Tim & Algoritma:
1. **Johanes Darren Yehuda** - Neural Network (MLP) Classifier
2. **Afrisya Dwiky Mauliddinka** - Support Vector Machine (SVM) Classifier
3. **Muhammad Hafizh Raharja** - K-Nearest Neighbors (KNN) Classifier

---

## 🛠️ Langkah 1: Persiapan Lingkungan

In [ ]:
import os
import sys
import json
import time
import shutil
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import SMOTE
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, ConfusionMatrixDisplay

warnings.filterwarnings('ignore')


# Buat folder penyimpanan model dan hasil grafik jika belum ada
os.makedirs('python/models', exist_ok=True)
os.makedirs('python/results/confusion_matrices', exist_ok=True)
print("Struktur direktori berhasil diinisialisasi: folder 'python/models/' dan 'python/results/' siap digunakan.")

### 📂 Unggah Dataset (`smart_plant_biosensor.csv`)

## 🌿 Langkah 2: Memuat & Membersihkan Data

Kita memuat berkas CSV, memeriksa ukuran tabel dan nama kolom, membuang kolom yang tidak relevan untuk prediksi (`Timestamp` dan `Plant_ID`), memeriksa data yang kosong (missing values), dan menganalisis sebaran kelas target `Plant_Health_Status`.

In [ ]:
# Memuat data
df = pd.read_csv('dataset/smart_plant_biosensor.csv')
print(f"Ukuran dataset: {df.shape[0]} baris, {df.shape[1]} kolom\n")

# Membersihkan data
columns_to_drop = ['Timestamp', 'Plant_ID']
existing_drops = [col for col in columns_to_drop if col in df.columns]
df_clean = df.drop(columns=existing_drops)
print(f"Kolom yang dibuang: {existing_drops}")

# Memeriksa missing values
missing = df_clean.isnull().sum()
if missing.sum() > 0:
    print(f"Nilai kosong ditemukan:\n{missing[missing > 0]}")
    df_clean = df_clean.dropna()
    print(f"Jumlah baris setelah menghapus NaN: {df_clean.shape[0]}")
else:
    print("Tidak ada nilai kosong dalam dataset.")

print("\nSebaran Kelas Target (Plant_Health_Status):")
print(df_clean['Plant_Health_Status'].value_counts())

df_clean.head()

## 📊 Langkah 3: Analisis Data Eksploratif (EDA)

Visualisasi fitur untuk memperoleh pemahaman mendalam tentang hubungan korelasi, distribusi nilai, serta frekuensi tiap kelas target.

In [ ]:
# 1. Heatmap Korelasi
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(12, 10))
corr_matrix = df_clean[numeric_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap='RdYlGn', center=0, fmt='.2f', linewidths=0.5, square=True)
plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('python/results/correlation_heatmap.png', dpi=150)
plt.close()

# 2. Distribusi Sebaran Kelas
plt.figure(figsize=(8, 5))
colors = ['#2ecc71', '#f39c12', '#e74c3c']
df_clean['Plant_Health_Status'].value_counts().plot(kind='bar', color=colors, edgecolor='black')
plt.title('Plant Health Status Distribution', fontsize=16, fontweight='bold')
plt.xlabel('Health Status')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('python/results/class_distribution.png', dpi=150)
plt.close()

# 3. Distribusi Fitur per Kelas (Boxplot)
fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes_flat = axes.flatten()
for i, col in enumerate(numeric_cols):
    if i < len(axes_flat):
        df_clean.boxplot(column=col, by='Plant_Health_Status', ax=axes_flat[i])
        axes_flat[i].set_title(col, fontsize=10, fontweight='bold')
        axes_flat[i].set_xlabel('')
# Sembunyikan axes yang tidak terpakai
for j in range(len(numeric_cols), len(axes_flat)):
    axes_flat[j].set_visible(False)
plt.suptitle('Feature Distributions by Plant Health Status', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('python/results/feature_distributions.png', dpi=150)
plt.close()

## ⚙️ Langkah 4: Pra-pemrosesan & Pembagian Data

Kita memisahkan fitur dari target, menyandikan label target ke nilai numerik (`LabelEncoder`), melakukan standarisasi skala fitur (`StandardScaler`), membagi dataset menjadi data training dan testing (**80/20 Stratified Split**), serta mengekspor scaler dan nama fitur untuk kebutuhan backend prediksi nanti.

In [ ]:
# Pisahkan fitur (X) dan target (y)
X = df_clean.drop(columns=['Plant_Health_Status'])
y = df_clean['Plant_Health_Status']

# Encode target menjadi angka numerik
le = LabelEncoder()
y_encoded = le.fit_transform(y)
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(f"Pemetaan Label Kelas: {label_mapping}")

# Analisis Seleksi Fitur (SelectKBest f_classif)
selector = SelectKBest(score_func=f_classif, k='all')
selector.fit(X, y_encoded)
scores_df = pd.DataFrame({
    'Feature': X.columns,
    'F_Score': selector.scores_,
    'P_Value': selector.pvalues_
}).sort_values('F_Score', ascending=False)
scores_df.to_csv('python/results/feature_scores.csv', index=False)

# Plot Skor F-Score untuk masing-masing Fitur
plt.figure(figsize=(10, 6))
colors_fs = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(scores_df)))
plt.barh(scores_df['Feature'], scores_df['F_Score'], color=colors_fs, edgecolor='black')
plt.xlabel('F-Score', fontsize=12)
plt.title('Feature Importance (F-Score)', fontsize=16, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('python/results/feature_importance.png', dpi=150)
plt.close()

# Standarisasi Skala Fitur
feature_names = list(X.columns)
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=feature_names)

# Pembagian data train/test secara Stratified (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

# Ekspor konfigurasi untuk digunakan pada backend prediksi web
joblib.dump(scaler, 'python/models/scaler.pkl')
joblib.dump(le, 'python/models/label_encoder.pkl')
with open('python/models/feature_names.json', 'w') as f:
    json.dump(feature_names, f)

print(f"\nJumlah data train: {X_train.shape[0]}")
print(f"Jumlah data test: {X_test.shape[0]}")
print("Berkas berhasil disimpan: scaler.pkl, label_encoder.pkl, feature_names.json")

## ⚖️ Langkah 5: Penanganan Data Imbalance (SMOTE)

Mengecek distribusi kelas pada data training dan menerapkan Synthetic Minority Over-sampling Technique (SMOTE) untuk menyeimbangkan jumlah sampel tiap kelas agar model tidak bias terhadap kelas mayoritas.

In [ ]:
print("=== Penanganan Imbalance Data (SMOTE) ===")

# Hitung distribusi sebelum SMOTE
unique_before, counts_before = np.unique(y_train, return_counts=True)
labels_before = [f"{le.classes_[u]} ({u})" for u in unique_before]

print("\nDistribusi kelas sebelum SMOTE:")
for u, c in zip(unique_before, counts_before):
    print(f"Kelas {le.classes_[u]} ({u}): {c} sampel")

# Terapkan SMOTE
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Hitung distribusi setelah SMOTE
unique_after, counts_after = np.unique(y_train_resampled, return_counts=True)
labels_after = [f"{le.classes_[u]} ({u})" for u in unique_after]

print("\nDistribusi kelas setelah SMOTE:")
for u, c in zip(unique_after, counts_after):
    print(f"Kelas {le.classes_[u]} ({u}): {c} sampel")

# Visualisasi Grafik Perbandingan
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors_before = ['#e74c3c', '#f39c12', '#2ecc71']

axes[0].bar(labels_before, counts_before, color=colors_before, edgecolor='black')
axes[0].set_title('Distribusi Kelas Sebelum SMOTE', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Kelas')
axes[0].set_ylabel('Jumlah Sampel')
for i, v in enumerate(counts_before):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

axes[1].bar(labels_after, counts_after, color='#3498db', edgecolor='black')
axes[1].set_title('Distribusi Kelas Setelah SMOTE', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Kelas')
axes[1].set_ylabel('Jumlah Sampel')
for i, v in enumerate(counts_after):
    axes[1].text(i, v + 20, str(v), ha='center', fontweight='bold')

plt.suptitle('Perbandingan Distribusi Kelas (Imbalance vs Balance)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('python/results/smote_distribution.png', dpi=150)
plt.close()

## 🤖 Langkah 6: Pelatihan Model Support Vector Machine (SVM)
**Pengembang**: Afrisya Dwiky Mauliddinka

Kita menjalankan pencarian grid (GridSearchCV) dengan 5-Fold Stratified Cross-Validation untuk parameter `C`, `kernel`, `gamma`, dan `degree` untuk mendapatkan akurasi terbaik, lalu mengevaluasi model pada data uji.

In [ ]:
print("=== PROSES PELATIHAN MODEL SVM ===")

param_grid_svm = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto', 0.1],
    'degree': [2]
}

cv_svm = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
svm_clf = SVC(probability=True, random_state=42)

start_time = time.time()
grid_svm = GridSearchCV(svm_clf, param_grid_svm, cv=cv_svm, scoring='accuracy', n_jobs=1, verbose=1)
grid_svm.fit(X_train_resampled, y_train_resampled)
elapsed_svm = time.time() - start_time

best_svm = grid_svm.best_estimator_
best_params_svm = grid_svm.best_params_
best_cv_svm = grid_svm.best_score_

y_pred_svm = best_svm.predict(X_test)
test_acc_svm = accuracy_score(y_test, y_pred_svm)

print(f"Parameter Terbaik: {best_params_svm}")
print(f"Akurasi Terbaik CV: {best_cv_svm:.4f}")
print(f"Akurasi Data Uji: {test_acc_svm:.4f}\n")
print(classification_report(y_test, y_pred_svm, target_names=le.classes_))

# Simpan Confusion Matrix
plt.figure(figsize=(6, 5))
cm_svm = confusion_matrix(y_test, y_pred_svm)
disp_svm = ConfusionMatrixDisplay(confusion_matrix=cm_svm, display_labels=le.classes_)
disp_svm.plot(cmap='Greens', values_format='d')
plt.title('SVM - Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('python/results/confusion_matrices/svm_confusion_matrix.png', dpi=150)
plt.close()

# Simpan Model & Hasil Metrik
joblib.dump(best_svm, 'python/models/svm_model.pkl')
svm_results = {
    'model': 'SVM',
    'author': 'Afrisya Dwiky Mauliddinka',
    'best_params': best_params_svm,
    'best_cv_accuracy': round(best_cv_svm, 4),
    'test_accuracy': round(test_acc_svm, 4),
    'classification_report': classification_report(y_test, y_pred_svm, target_names=le.classes_, output_dict=True),
    'training_time_seconds': round(elapsed_svm, 1)
}
with open('python/results/svm_results.json', 'w') as f:
    json.dump(svm_results, f, indent=2)

## 🤖 Langkah 7: Pelatihan Model K-Nearest Neighbors (KNN)
**Pengembang**: Muhammad Hafizh Raharja

Kita mengoptimalkan parameter KNN pada `n_neighbors`, `weights`, `metric`, dan tingkat pangkat minkowski `p` menggunakan GridSearchCV.

In [ ]:
print("=== PROSES PELATIHAN MODEL KNN ===")

param_grid_knn = {
    'n_neighbors': [3, 5, 7, 9, 11, 13, 15],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan'],
    'p': [1, 2]
}

cv_knn = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
knn_clf = KNeighborsClassifier()

start_time = time.time()
grid_knn = GridSearchCV(knn_clf, param_grid_knn, cv=cv_knn, scoring='accuracy', n_jobs=1, verbose=1)
grid_knn.fit(X_train_resampled, y_train_resampled)
elapsed_knn = time.time() - start_time

best_knn = grid_knn.best_estimator_
best_params_knn = grid_knn.best_params_
best_cv_knn = grid_knn.best_score_

y_pred_knn = best_knn.predict(X_test)
test_acc_knn = accuracy_score(y_test, y_pred_knn)

print(f"Parameter Terbaik: {best_params_knn}")
print(f"Akurasi Terbaik CV: {best_cv_knn:.4f}")
print(f"Akurasi Data Uji: {test_acc_knn:.4f}\n")
print(classification_report(y_test, y_pred_knn, target_names=le.classes_))

# Kurva perbandingan nilai K vs Akurasi Data Uji
k_range = range(1, 21)
k_scores = []
for k in k_range:
    knn_temp = KNeighborsClassifier(
        n_neighbors=k,
        weights=best_params_knn.get('weights', 'uniform'),
        metric=best_params_knn.get('metric', 'euclidean')
    )
    knn_temp.fit(X_train_resampled, y_train_resampled)
    k_scores.append(knn_temp.score(X_test, y_test))

plt.figure(figsize=(8, 5))
plt.plot(k_range, k_scores, 'bo-', linewidth=2, markersize=8)
plt.fill_between(k_range, k_scores, alpha=0.1, color='blue')
plt.xlabel('Jumlah Neighbors (K)', fontsize=12)
plt.ylabel('Akurasi', fontsize=12)
plt.title('KNN - K Value vs Accuracy', fontsize=14, fontweight='bold')
plt.xticks(k_range)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('python/results/knn_k_vs_accuracy.png', dpi=150)
plt.close()

# Simpan Confusion Matrix
plt.figure(figsize=(6, 5))
cm_knn = confusion_matrix(y_test, y_pred_knn)
disp_knn = ConfusionMatrixDisplay(confusion_matrix=cm_knn, display_labels=le.classes_)
disp_knn.plot(cmap='Blues', values_format='d')
plt.title('KNN - Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('python/results/confusion_matrices/knn_confusion_matrix.png', dpi=150)
plt.close()

# Simpan Model & Hasil Uji
joblib.dump(best_knn, 'python/models/knn_model.pkl')
knn_results = {
    'model': 'KNN',
    'author': 'Muhammad Hafizh Raharja',
    'best_params': best_params_knn,
    'best_cv_accuracy': round(best_cv_knn, 4),
    'test_accuracy': round(test_acc_knn, 4),
    'classification_report': classification_report(y_test, y_pred_knn, target_names=le.classes_, output_dict=True),
    'training_time_seconds': round(elapsed_knn, 1)
}
with open('python/results/knn_results.json', 'w') as f:
    json.dump(knn_results, f, indent=2)

## 🤖 Langkah 8: Pelatihan & Visualisasi Neural Network (MLP)
**Pengembang**: Johanes Darren Yehuda

Kita melatih model **Artificial Neural Network** menggunakan `MLPClassifier`. Kita mengoptimasi arsitektur lapisan tersembunyi (`hidden_layer_sizes`), fungsi aktivasi, dan *learning rate* untuk mendapatkan performa generalisasi yang lebih realistis.

In [ ]:
print("=== PROSES PELATIHAN MODEL NEURAL NETWORK (MLP) ===")

# Tentukan grid parameter untuk Neural Network
param_grid_nn = {
    'hidden_layer_sizes': [(32,), (64, 32), (16, 16, 16)],
    'activation': ['relu', 'tanh'],
    'solver': ['adam'],
    'alpha': [0.0001, 0.001],
    'learning_rate_init': [0.001, 0.01]
}

cv_nn = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
mlp_clf = MLPClassifier(random_state=42, max_iter=500)

start_time = time.time()
grid_nn = GridSearchCV(mlp_clf, param_grid_nn, cv=cv_nn, scoring='accuracy', n_jobs=1, verbose=1)
grid_nn.fit(X_train_resampled, y_train_resampled)
elapsed_nn = time.time() - start_time

best_nn = grid_nn.best_estimator_
best_params_nn = grid_nn.best_params_
best_cv_nn = grid_nn.best_score_

y_pred_nn = best_nn.predict(X_test)
test_acc_nn = accuracy_score(y_test, y_pred_nn)

print(f"Parameter Terbaik: {best_params_nn}")
print(f"Akurasi Terbaik CV: {best_cv_nn:.4f}")
print(f"Akurasi Data Uji: {test_acc_nn:.4f}\n")
print(classification_report(y_test, y_pred_nn, target_names=le.classes_))

# 1. Visualisasi Loss Curve
plt.figure(figsize=(8, 5))
plt.plot(best_nn.loss_curve_)
plt.title('Neural Network - Loss Curve', fontsize=14, fontweight='bold')
plt.xlabel('Iterations')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('python/results/nn_loss_curve.png', dpi=150)
plt.close()

# 2. Analisis Koefisien (Analog Feature Importance)
weights_input = np.abs(best_nn.coefs_[0]).mean(axis=1)
feature_imp_nn = pd.DataFrame({
    'Feature': feature_names,
    'Importance': weights_input
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
colors_nn = plt.cm.Purples(np.linspace(0.4, 0.9, len(feature_imp_nn)))
plt.barh(feature_imp_nn['Feature'], feature_imp_nn['Importance'], color=colors_nn, edgecolor='black')
plt.xlabel('Mean Absolute Weight (Input Layer)', fontsize=12)
plt.title('Neural Network - Feature Sensitivity', fontsize=16, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('python/results/nn_feature_importance.png', dpi=150)
plt.close()

# 3. Confusion Matrix
plt.figure(figsize=(6, 5))
cm_nn = confusion_matrix(y_test, y_pred_nn)
disp_nn = ConfusionMatrixDisplay(confusion_matrix=cm_nn, display_labels=le.classes_)
disp_nn.plot(cmap='Purples', values_format='d')
plt.title('Neural Network - Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('python/results/confusion_matrices/nn_confusion_matrix.png', dpi=150)
plt.close()

# Simpan Model & Hasil
joblib.dump(best_nn, 'python/models/nn_model.pkl')
nn_results = {
    'model': 'Neural Network',
    'author': 'Johanes Darren Yehuda',
    'best_params': {k: str(v) for k, v in best_params_nn.items()},
    'best_cv_accuracy': round(best_cv_nn, 4),
    'test_accuracy': round(test_acc_nn, 4),
    'classification_report': classification_report(y_test, y_pred_nn, target_names=le.classes_, output_dict=True),
    'training_time_seconds': round(elapsed_nn, 1)
}
with open('python/results/nn_results.json', 'w') as f:
    json.dump(nn_results, f, indent=2)

## 📊 Langkah 9: Ringkasan & Perbandingan Master

Kita mengumpulkan hasil evaluasi dari ketiga model, memvisualisasikan performa akurasi, nilai F1-score tiap kelas, serta perbandingan durasi waktu komputasi pelatihan model.

In [ ]:
all_results = [svm_results, knn_results, nn_results]

models = [r['model'] for r in all_results]
cv_accuracies = [r['best_cv_accuracy'] for r in all_results]
test_accuracies = [r['test_accuracy'] for r in all_results]
train_times = [r['training_time_seconds'] for r in all_results]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors_bar = ['#2ecc71', '#3498db', '#9b59b6']
x_pos = np.arange(len(models))
width = 0.35

axes[0].bar(x_pos - width/2, cv_accuracies, width, label='CV Accuracy', color=colors_bar, alpha=0.7, edgecolor='black')
axes[0].bar(x_pos + width/2, test_accuracies, width, label='Test Accuracy', color=colors_bar, edgecolor='black')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Model Accuracy Comparison', fontsize=12, fontweight='bold')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(models)
axes[0].legend()
axes[0].set_ylim(0, 1.1)

classes = list(all_results[0]['classification_report'].keys())
classes = [c for c in classes if c not in ['accuracy', 'macro avg', 'weighted avg']]
for c_idx, cls_lbl in enumerate(classes):
    f1_scores = [r['classification_report'][cls_lbl]['f1-score'] for r in all_results]
    axes[1].bar(x_pos + (c_idx - 1) * width * 0.8, f1_scores, width * 0.8, label=cls_lbl, alpha=0.85, edgecolor='black')

axes[1].set_title('F1-Score per Class', fontsize=12, fontweight='bold')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(models)
axes[1].legend()

axes[2].bar(models, train_times, color=colors_bar, edgecolor='black')
axes[2].set_ylabel('Time (s)')
axes[2].set_title('Training Time Comparison', fontsize=12, fontweight='bold')

plt.suptitle('Model Comparison (SVM vs KNN vs Neural Network)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('python/results/model_comparison.png', dpi=150, bbox_inches='tight')
plt.close()

# Save comparison JSON for backend API
best_model = max(all_results, key=lambda x: x['test_accuracy'])
comparison = {
    'models': all_results,
    'best_model': {
        'name': best_model['model'],
        'test_accuracy': best_model['test_accuracy'],
        'author': best_model['author']
    }
}
with open('python/results/model_comparison.json', 'w') as f:
    json.dump(comparison, f, indent=2)

# Salin semua gambar hasil ke public/results/ agar bisa ditampilkan di web
copy_results_to_public()
print("\n=== PIPELINE SELESAI ===")
print("Semua model, metrik, dan visualisasi berhasil dibuat.")


def copy_results_to_public():

Salin semua file PNG dan JSON dari python/results/ ke public/results/

In [ ]:
src_dir = 'python/results'
dst_dir = 'public/results'
os.makedirs(dst_dir, exist_ok=True)
os.makedirs(os.path.join(dst_dir, 'confusion_matrices'), exist_ok=True)

for root, dirs, files_list in os.walk(src_dir):
    for file in files_list:
        if file.endswith(('.png', '.json', '.csv')):
            src = os.path.join(root, file)
            rel_path = os.path.relpath(src, src_dir)
            dest = os.path.join(dst_dir, rel_path)
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            shutil.copy2(src, dest)
print(f"Semua hasil berhasil disalin ke {dst_dir}/")